# Adapter Hizli Testi

Bu notebook tek bir adapter'i dogrudan yukler, metadata bilgisini okur, tek goruntu tahmini yapar ve isterseniz bir klasor uzerinde hizli saglik kontrolu calistirir.

Notebook proje klasoru ve yaygin Drive koklerinde adapter bundle arar. Elle yol vermek de desteklenir.

Manuel giris alanlari:
- `ADAPTER_DIR`: adapter klasoru, export klasoru, telemetry run klasoru veya `adapter_meta.json`
- `ADAPTER_ROOT`: `models/adapters/` gibi crop klasorlerinin ebeveyni
- `IMAGE_PATH`: tek bir goruntu
- `BATCH_IMAGE_DIR`: bir klasor dolusu goruntu

Ayni oturum icinde tekrar denemek icin:
- `FORCE_ADAPTER_RESCAN = True` yapip adapter kesif hucresini tekrar calistirin
- tek goruntu hucresi varsayilan olarak yeni upload ister
- mevcut upload'i korumak icin `REUSE_EXISTING_IMAGE_PATH = True` yapin
- hemen yeni upload zorlamak icin `FORCE_IMAGE_UPLOAD = True` yapin


In [ ]:
import os
import sys
import subprocess
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
GITHUB_TOKEN_NAMES = ('GH_TOKEN', 'GITHUB_TOKEN')
COMMON_REPO_CANDIDATES = (
    Path('/content/bitirme projesi'),
    CLONE_TARGET,
    Path('/content/aads_ulora'),
)

def _is_repo_root(path: Path) -> bool:
    return (path / 'src').is_dir() and (path / 'config').is_dir() and (path / 'scripts').is_dir()

def _running_in_colab_inline() -> bool:
    try:
        import google.colab  # noqa: F401
    except Exception:
        return False
    return True

def _resolve_colab_secret_inline(secret_name: str) -> str:
    if not _running_in_colab_inline():
        return ''
    try:
        from google.colab import userdata
        return str(userdata.get(secret_name) or '').strip()
    except Exception:
        return ''

def _resolve_github_token_inline() -> str:
    for env_name in GITHUB_TOKEN_NAMES:
        token = str(os.environ.get(env_name, '')).strip()
        if token:
            os.environ.setdefault('GH_TOKEN', token)
            return token
    for secret_name in GITHUB_TOKEN_NAMES:
        token = _resolve_colab_secret_inline(secret_name)
        if token:
            os.environ['GH_TOKEN'] = token
            return token
    return ''

def _build_repo_access_url(repo_url: str, token: str) -> str:
    if not token:
        return repo_url
    parsed = urlsplit(str(repo_url or '').strip())
    if parsed.scheme != 'https' or not parsed.netloc:
        return repo_url
    netloc = parsed.netloc.split('@', 1)[-1]
    return urlunsplit((parsed.scheme, f'{token}@{netloc}', parsed.path, parsed.query, parsed.fragment))

def _find_repo_root_inline() -> Path | None:
    for raw in (os.environ.get('AADS_REPO_ROOT'), os.environ.get('REPO_ROOT')):
        if not raw:
            continue
        candidate = Path(raw).expanduser().resolve()
        if _is_repo_root(candidate):
            return candidate
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if _is_repo_root(candidate):
            return candidate
    for candidate in COMMON_REPO_CANDIDATES:
        if _is_repo_root(candidate):
            return candidate
    if CLONE_TARGET.exists() and any(CLONE_TARGET.iterdir()):
        for child in CLONE_TARGET.iterdir():
            if child.is_dir() and _is_repo_root(child):
                return child
    return None

def _ensure_repo_root_for_bootstrap() -> Path:
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    clone_url = _build_repo_access_url(REPO_URL, _resolve_github_token_inline())
    CLONE_TARGET.parent.mkdir(parents=True, exist_ok=True)
    completed = subprocess.run(
        ['git', 'clone', '--depth', '1', clone_url, str(CLONE_TARGET)],
        check=False,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if completed.stdout:
        print(completed.stdout)
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    raise RuntimeError(
        'Repo bootstrap basarisiz oldu. Mevcut bir checkout icin AADS_REPO_ROOT ayarlayin veya '
        'private GitHub auto-clone icin GH_TOKEN/GITHUB_TOKEN saglayin.'
    )

BOOTSTRAP_ROOT = _ensure_repo_root_for_bootstrap()
os.chdir(BOOTSTRAP_ROOT)
if str(BOOTSTRAP_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOTSTRAP_ROOT))

from scripts.colab_repo_bootstrap import (
    collect_notebook_access_report,
    install_colab_requirements,
    print_notebook_access_report,
    resolve_repo_root,
    running_in_colab,
)

ROOT = resolve_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if running_in_colab():
    install_colab_requirements(ROOT / 'requirements_colab.txt', in_colab=True)

from src.core.config_manager import get_config

CONFIG_FOR_ACCESS = get_config(environment='colab')
BACKBONE_MODEL_NAME = str(dict(dict(CONFIG_FOR_ACCESS.get('training', {})).get('continual', {})).get('backbone', {}).get('model_name', '')).strip()
ACCESS_REPORT = collect_notebook_access_report(
    repo_root=ROOT,
    hf_model_ids=[BACKBONE_MODEL_NAME] if BACKBONE_MODEL_NAME else [],
)
print_notebook_access_report(ACCESS_REPORT, print_fn=print)
if BACKBONE_MODEL_NAME:
    print(f"[KONTROL] Varsayilan backbone modeli: {BACKBONE_MODEL_NAME}")
print(ROOT)


In [ ]:
print('[KONTROL] Access/update check was already executed in the first code cell.')
if BACKBONE_MODEL_NAME:
    print(f"[KONTROL] Varsayilan backbone modeli: {BACKBONE_MODEL_NAME}")


In [ ]:
import json
from collections import Counter
from pathlib import Path

from IPython.display import display

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import ipywidgets as widgets
except Exception:
    widgets = None

from scripts.colab_adapter_smoke_test import (
    discover_adapter_candidates,
    load_adapter_summary,
    predict_image_folder,
    predict_single_image,
)

# Dogrudan smoke-test ayarlari.
CROP_NAME = None
# Ornek: 'tomato'. None birakirsaniz bulunan tum adapterler listelenir.

ADAPTER_DIR = None
# Kabul edilen ornekler:
# - .../continual_sd_lora_adapter/
# - .../artifacts/adapter_export/
# - outputs/colab_notebook_training/
# - .../telemetry/<RUN_ID>/ veya .../telemetry/<RUN_ID>/artifacts/
# - .../adapter_meta.json

ADAPTER_ROOT = None
# Normalde models/adapters/ gibi crop klasorlerinin ebeveyni olur.

SEARCH_ROOTS = [
    ROOT,
    ROOT / 'outputs',
    ROOT / 'models',
    ROOT / 'models' / 'adapters',
]
SELECTED_ADAPTER_INDEX = 0
FORCE_ADAPTER_RESCAN = False
IMAGE_PATH = None
FORCE_IMAGE_UPLOAD = False
# Tek goruntu dosyasi
BATCH_IMAGE_DIR = None
# Opsiyonel klasor testi icin goruntu klasoru
DEVICE = 'cuda'
CONFIG_ENV = 'colab'
REUSE_EXISTING_IMAGE_PATH = False

print(f'Cihaz={DEVICE} config_env={CONFIG_ENV}')
print('ADAPTER_DIR verirseniz kesif atlanir. Aksi halde notebook SEARCH_ROOTS altinda adapter arar.')
print('Kesif kokleri:')
for root in SEARCH_ROOTS:
    print(f'- {root}')
print('ADAPTER_ROOT crop klasorlerinin ebeveyni olmali, IMAGE_PATH tek dosya olmali, BATCH_IMAGE_DIR ise tek klasor olmali.')
print('Adapter listesini sifirdan kurmak icin FORCE_ADAPTER_RESCAN=True yapin.')
print('Tek goruntu hucresi varsayilan olarak yeni upload ister. Mevcut dosyayi korumak icin REUSE_EXISTING_IMAGE_PATH=True yapin.')


In [ ]:
adapter_candidates = []
adapter_selector = None

if FORCE_ADAPTER_RESCAN:
    ADAPTER_DIR = None
    print('FORCE_ADAPTER_RESCAN=True, adapter listesi yeniden kuruluyor.')

if ADAPTER_DIR is not None:
    SELECTED_ADAPTER_DIR = str(Path(ADAPTER_DIR))
    print(f'Elle verilen ADAPTER_DIR kullaniliyor: {SELECTED_ADAPTER_DIR}')
else:
    adapter_candidates = discover_adapter_candidates(SEARCH_ROOTS, crop_name=CROP_NAME)
    if not adapter_candidates:
        raise FileNotFoundError(
            'SEARCH_ROOTS altinda adapter bulunamadi. Gerekirse ADAPTER_DIR degerini elle verin veya SEARCH_ROOTS listesini guncelleyin.'
        )

    print('Bulunan adapterler:')
    for index, candidate in enumerate(adapter_candidates):
        print(f'[{index}] {candidate["display_name"]}')

    if widgets is not None:
        adapter_selector = widgets.Dropdown(
            options=[(candidate['display_name'], index) for index, candidate in enumerate(adapter_candidates)],
            value=min(SELECTED_ADAPTER_INDEX, len(adapter_candidates) - 1),
            description='Adapter:',
            layout=widgets.Layout(width='95%'),
        )
        display(adapter_selector)
        print('Dropdown uzerinden adapter secin, sonra siradaki hucreyi calistirin.')
    else:
        print('ipywidgets yok. SELECTED_ADAPTER_INDEX degerini istediginiz adaptere ayarlayip sonraki hucreyi calistirin.')


In [ ]:
if ADAPTER_DIR is None:
    selected_index = adapter_selector.value if adapter_selector is not None else SELECTED_ADAPTER_INDEX
    selected_candidate = adapter_candidates[int(selected_index)]
    ADAPTER_DIR = selected_candidate['adapter_dir']
    if CROP_NAME is None:
        CROP_NAME = selected_candidate.get('crop_name')
    print(json.dumps(selected_candidate, indent=2))
    FORCE_ADAPTER_RESCAN = False

summary = load_adapter_summary(
    CROP_NAME,
    adapter_dir=ADAPTER_DIR,
    adapter_root=ADAPTER_ROOT,
    config_env=CONFIG_ENV,
    device=DEVICE,
)

CROP_NAME = summary['crop_name']
ADAPTER_DIR = summary['resolved_adapter_dir']
print(f"Cozulen crop_name={CROP_NAME}")
print(f"Cozulen adapter_dir={ADAPTER_DIR}")

print(json.dumps(summary, indent=2))


In [ ]:
request_new_upload = running_in_colab() and not bool(REUSE_EXISTING_IMAGE_PATH)
adapter_summary = globals().get('summary')
if (ADAPTER_DIR is None or CROP_NAME is None) and isinstance(adapter_summary, dict):
    ADAPTER_DIR = adapter_summary.get('resolved_adapter_dir', ADAPTER_DIR)
    CROP_NAME = adapter_summary.get('crop_name', CROP_NAME)

if ADAPTER_DIR is None and CROP_NAME is None:
    raise RuntimeError(
        'Once adapter secim/ozet hucresini calistirin ki Notebook 3 ADAPTER_DIR ve CROP_NAME degerlerini cozebilsin.'
    )

if FORCE_IMAGE_UPLOAD:
    IMAGE_PATH = None
    print('FORCE_IMAGE_UPLOAD=True, yeni goruntu yuklemesi isteniyor.')
elif request_new_upload:
    IMAGE_PATH = None
    print('REUSE_EXISTING_IMAGE_PATH=False, bu calisma icin yeni goruntu yuku istegi acildi.')

if IMAGE_PATH is None:
    if not running_in_colab():
        raise ValueError('Colab disinda calisiyorsaniz IMAGE_PATH degerini tek bir goruntu dosyasina ayarlayin.')
    from google.colab import files
    uploaded = files.upload()
    IMAGE_PATH = next(iter(uploaded.keys()))
    FORCE_IMAGE_UPLOAD = False

IMAGE_PATH = str(Path(IMAGE_PATH))
print(f'Kullanilan IMAGE_PATH: {IMAGE_PATH}')

single_result = predict_single_image(
    IMAGE_PATH,
    CROP_NAME,
    adapter_dir=ADAPTER_DIR,
    adapter_root=ADAPTER_ROOT,
    config_env=CONFIG_ENV,
    device=DEVICE,
)

print(json.dumps(single_result, indent=2))


In [ ]:
if not BATCH_IMAGE_DIR:
    print('Opsiyonel klasor testi icin BATCH_IMAGE_DIR degerini bir goruntu klasorune ayarlayin.')
else:
    rows = predict_image_folder(
        BATCH_IMAGE_DIR,
        CROP_NAME,
        adapter_dir=ADAPTER_DIR,
        adapter_root=ADAPTER_ROOT,
        config_env=CONFIG_ENV,
        device=DEVICE,
    )
    if pd is not None:
        df = pd.DataFrame(rows)
        display(df)
    else:
        print(rows)

    predicted_counts = Counter(row['predicted_class'] for row in rows if row.get('predicted_class'))
    ood_count = sum(1 for row in rows if row.get('is_ood') is True)
    error_count = sum(1 for row in rows if row.get('error'))

    print('predicted_class_counts:', dict(predicted_counts))
    print('ood_count:', ood_count)
    print('error_count:', error_count)
